# 허깅페이스에서_모델받아_다국어번역_서비스만들기

In [5]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "이것은 m2m 모델로 만든 번역기 입니다"
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# translate Hindi to French
tokenizer.src_lang = "ko"
encoded_hi = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en"))
result1 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result1)
# => "La vie est comme une boîte de chocolat."

# translate Chinese to English
tokenizer.src_lang = "ko"
encoded_zh = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("ja"))
result2 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result2)
# => "Life is like a box of chocolate."


['This is a translator made with a m2m model.']
['これはm2mモデルで作られた翻訳器です。']


In [6]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import torch

# 모델 및 토크나이저 로드
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# 지원 언어 코드와 이름 매핑
language_dict = {
    "Korean (ko)": "ko",
    "English (en)": "en",
    "Japanese (ja)": "ja",
    "Chinese (zh)": "zh",
    "French (fr)": "fr",
    "German (de)": "de",
    "Spanish (es)": "es"
}

# 번역 함수
def translate_text(text, source_lang_name, target_lang_name):
    source_lang = language_dict[source_lang_name]
    target_lang = language_dict[target_lang_name]

    tokenizer.src_lang = source_lang
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(
        **encoded, forced_bos_token_id=tokenizer.get_lang_id(target_lang)
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# Gradio UI 구성
iface = gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(label="Input Text", placeholder="번역할 문장을 입력하세요."),
        gr.Dropdown(choices=list(language_dict.keys()), label="Source Language", value="Korean (ko)"),
        gr.Dropdown(choices=list(language_dict.keys()), label="Target Language", value="English (en)")
    ],
    outputs=gr.Textbox(label="Translated Text"),
    title="다국어 번역기 (M2M100)",
    description="Facebook M2M100 모델을 사용한 번역기입니다. 원하는 언어를 선택하세요."
)

iface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
pip install datasets soundfile